In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
pd.set_option("display.max_columns", None)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import numpy as np

from modelling import *

In [ ]:
telco = pd.read_parquet("../data/clean_telco.parquet")
telco["SeniorCitizen"] = telco["SeniorCitizen"].map({0: "No", 1: "Yes"})
telco["Churn"] = telco["Churn"].map({"No": 0, "Yes": 1})
telco.drop(columns = ["customerID"], inplace = True)

In [ ]:
target = "Churn"

numerical_features = (
    telco
    .select_dtypes(
        include = "number"
    )
    .drop(
        columns = [
            "Churn"
        ]
    )
    .columns.tolist()
)

binary_features = (
    telco
    .loc[:, telco.nunique() == 2]
    .select_dtypes(
        include = "str"
    )
    .columns.tolist()
)

categorical_features = (
    telco
    .loc[:, telco.nunique() > 2]
    .select_dtypes(
        include = "str"
    )
    .columns.tolist()
)

# potentially possible to add `Contract` with the binary features
# we can assign an order to the categories with `Month-to-month` as 0
# `One year` as 1, `Two year` as 2

In [ ]:
model = LogisticRegression(
    random_state = 123,
    class_weight = "balanced"
)

logistic_regression = ModelTraining(
    model = model,
    data = telco,
    target = target,
    numerical_features = numerical_features,
    binary_features = binary_features,
    categorical_features = categorical_features
)

# testing with class weight:
# None -> churn recall 52.7%
# balanced -> churn recall 79.7%

In [ ]:
model = RandomForestClassifier(
    random_state = 123,
    class_weight = "balanced"
)

random_forest = ModelTraining(
    model = model,
    data = telco,
    target = target,
    numerical_features = numerical_features,
    binary_features = binary_features,
    categorical_features = categorical_features
)

In [ ]:
scale_pos_weight = len(telco[telco["Churn"] == 0]) / len(telco[telco["Churn"] == 1])

model = XGBClassifier(
    random_state = 123,
    scale_pos_weight = scale_pos_weight
)

xgboost = ModelTraining(
    model = model,
    data = telco,
    target = target,
    numerical_features = numerical_features,
    binary_features = binary_features,
    categorical_features = categorical_features
)

In [ ]:
lr_params = {}

rf_params = {
    "model__n_estimators": np.arange(100, 501, 50),
    "model__max_depth": np.arange(3, 11, 1),
    "model__min_samples_split": np.arange(2, 21, 3),
    "model__min_samples_leaf": np.arange(1, 23, 3),
    "model__max_features": ["sqrt", "log2", 0.25, 0.5]
}

xgb_params = {
    "model__n_estimators": np.arange(100, 501, 50),
    "model__learning_rate": np.arange(0.001, 1.05, 0.05),
    "model__max_depth": np.arange(3, 11, 1),
    "model__colsample_bytree": np.arange(0.1, 1.1, 0.1)
}


models = {
    "logistic regression": (logistic_regression, lr_params),
    "random forest": (random_forest, rf_params),
    "xgboost": (xgboost, xgb_params)
}

for name, (model, params) in models.items():
    print(f"MODEL: {name}")
    print("--------------------------\n")
    model.train()
    model.evaluate()

    if name != "logistic regression":
        print("\nOPTIMISE")
        print("--------------------------\n")
        model.optimise(
            params = params,
            scoring = "f1"
        )
        model.evaluate()

    print("\nTHRESHOLD")
    print("--------------------------\n")
    model.threshold(
        scoring = "f1"
    )
    model.evaluate(
        report = True
    )

    print("--------------------------")